# Instruction Finetuning de modelo LLM base

In [1]:
!pip install torch
!pip install tensorflow

In [2]:
import os

# 1. Configurar Keras con Backend de TensorFlow
os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf
import keras
import keras_hub

# Activar crecimiento de memoria para evitar errores OOM
'''
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
'''

"\ngpus = tf.config.list_physical_devices('GPU')\nif gpus:\n    for gpu in gpus:\n        tf.config.experimental.set_memory_growth(gpu, True)\n"

# Training dataset

In [6]:
import json
import tensorflow as tf

def cargar_y_formatear_chatml(jsonl_path):
    textos_formateados = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            string_conversacion = ""

            # Unimos los mensajes usando los tokens especiales de Qwen (ChatML)
            for msg in data["messages"]:
                role = msg["role"]
                content = msg["content"]
                string_conversacion += f"<|im_start|>{role}\n{content}<|im_end|>\n"

            textos_formateados.append(string_conversacion)
    print(f'Casos procesados: {len(textos_formateados)}')
    # Creamos el dataset de TensorFlow que KerasHub entiende de forma nativa
    dataset = tf.data.Dataset.from_tensor_slices(textos_formateados)

    # CRUCIAL: Duplicamos el texto para que actúe como entrada (x) y objetivo (y)
    # KerasHub se encargará de tokenizar y desfasar un token de forma interna
    dataset = dataset.map(lambda x: (x, x))

    # por lo que solo necesitamos mezclar y empaquetar en batches.
    # Hay que mezclar entre epocas de entrenamiento
    return dataset.shuffle(buffer_size=1000).batch(1)

# Reemplaza esto antes de tu código de entrenamiento:
train_dataset = cargar_y_formatear_chatml("corpus_chatbot_guia_metodos_numericos.jsonl")
print(f"Se cargaron {len(train_dataset)} casos para entrenar")

Casos procesados: 30
Se cargaron 30 casos para entrenar


#Configuración del modelo

In [21]:
#import kagglehub
#kagglehub.login()

In [20]:
#from huggingface_hub import login
#login(add_to_git_credential=True) # Almacena de forma segura la credencial

In [19]:
# 1. Clear any leftover backend memory from the previous crash
keras.backend.clear_session()
#https://keras.io/keras_hub/presets/
# Use float16 (T4 does not support bfloat16)
#keras.config.set_floatx("float32")
keras.mixed_precision.set_global_policy("mixed_float16")
#model_preset = "gemma3_instruct_1b"
#causal_lm = keras_hub.models.Gemma3CausalLM.from_preset(
#    model_preset
#)
model_preset = "qwen3_1.7b_en"
causal_lm = keras_hub.models.Qwen3CausalLM.from_preset(
    model_preset
)
#causal_lm = keras_hub.models.GPT2CausalLM.from_preset(
#    "gpt2_medium_en")

ValueError: Unknown preset identifier. A preset must be a one of:
1) a built-in preset identifier like `'bert_base_en'`
2) a Kaggle Models handle like `'kaggle://keras/bert/keras/bert_base_en'`
3) a Hugging Face handle like `'hf://username/bert_base_en'`
4) a Modelscope handle like `'modelscope://username/bert_base_en'`
5) a path to a local preset directory like `'./bert_base_en'`
Use `print(cls.presets.keys())` to view all built-in presets for API symbol `cls`.
Received: preset='qwen3_1.7b'

In [ ]:
# Enable QLoRA with higher, more accurate settings
# Since sequence_length is capped, we have plenty of VRAM for int8 and rank=8.
#causal_lm.quantize("int8")
causal_lm.preprocessor.sequence_length = 256 #Limit context for OOM
causal_lm.backbone.enable_lora(rank=16)

In [ ]:
# Compilar el modelo
causal_lm.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=2e-5, weight_decay=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()]
)

#Entrenamiento

In [ ]:
# Ajustar el modelo (Aumentar epochs según el tamaño real de tus datos)
causal_lm.fit(
    train_dataset,
    epochs=20, verbose=1
)

Epoch 1/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 103s 381ms/step - loss: 1.7314 - sparse_categorical_accuracy: 0.4645
Epoch 2/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 16s 393ms/step - loss: 1.5869 - sparse_categorical_accuracy: 0.4685
Epoch 3/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 18s 420ms/step - loss: 1.4284 - sparse_categorical_accuracy: 0.4717
Epoch 4/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 20s 403ms/step - loss: 1.2994 - sparse_categorical_accuracy: 0.5023
Epoch 5/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 18s 424ms/step - loss: 1.1762 - sparse_categorical_accuracy: 0.5395
Epoch 6/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 401ms/step - loss: 1.0535 - sparse_categorical_accuracy: 0.5892
Epoch 7/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 18s 421ms/step - loss: 0.8902 - sparse_categorical_accuracy: 0.6293
Epoch 8/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 403ms/step - loss: 0.6978 - sparse_categorical_accuracy: 0.7188
Epoch 9/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 401ms/step - loss: 0.6217 - sparse_categorical_accuracy: 0.7371
Epoch 10/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 18s 42

In [ ]:
print("--- COMPILACIÓN DEL MODELO ---")
causal_lm.summary()

--- COMPILACIÓN DEL MODELO ---


Preprocessor: "qwen3_causal_lm_preprocessor_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ qwen3_tokenizer (Qwen3Tokenizer)                              │                      Vocab size: 151,669 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "qwen3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ qwen3_backbone                │ (None, None, 2048)        │   1,742,709,760 │ padding_mask[0][0],        │
│ (Qwen3Backbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 151936)      │     311,164,928 │ qwen3_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,786,979,334 (6.66 GB)

 Trainable params: 22,134,784 (84.44 MB)

 Non-trainable params: 1,720,574,976 (6.41 GB)

 Optimizer params: 44,269,574 (168.88 MB)

In [ ]:
causal_lm.save("qwen3_ft_tuned.keras")

In [ ]:
causal_lm.backbone.save_as_hf("ft_tuned_hf_format")

# Testing

In [ ]:
def test_model_tutor(pregunta_estudiante):
    # 1. Construir el prompt EXACTO con el formato ChatML que usamos en el entrenamiento
    prompt = (
        f"<|im_start|>system\n"
        f"Eres un chatbot del curso de Métodos Numéricos en Ingenieria. Respondes preguntas de estudiantes sobre la guia de estudio.<|im_end|>\n"
        f"<|im_start|>user\n"
        f"{pregunta_estudiante}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    # 2. Generar la respuesta del modelo
    # Subimos un poco max_length por si el tutor se extiende en su explicación socrática
    response = causal_lm.generate(prompt, max_length=512)

    # 3. Limpieza: Extraer solo lo que respondió el asistente (para no reimprimir todo el prompt)
    try:
        assistant_reply = response.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()
    except Exception:
        assistant_reply = response # Backup por si los tokens de parada fallan

    return assistant_reply

# --- Casos de Prueba Reales para tu Tutor Socrático ---
test_cases = [
    # Caso 1: La fórmula es r = (a + b) / 2, donde 'a' y 'b' son los extremos del intervalo.
    "Profe, ¿Cuál es la fórmula de recurrencia del método de Bisección?",

    # Caso 2: Son conjuntos de órdenes que constituyen todo un proceso identificadas con un nombre particular. Pueden recibir o entregar argumentos de entrada y salida.
    "¿Para qué sirven los Subprogramas?"
]

# --- Ejecución de las pruebas ---
print("=== EVALUANDO RESPUESTAS DEL TUTOR SOCRÁTICO ===")
for estudiante_input in test_cases:
    reply = test_model_tutor(estudiante_input)
    print(f"\n[ESTUDIANTE]: {estudiante_input}")
    print(f"[TUTOR IA]:   {reply}")
    print("-" * 80)

=== EVALUANDO RESPUESTAS DEL TUTOR SOCRÁTICO ===
generated token ids =  Tensor("strided_slice_28:0", shape=(512,), dtype=int32)

[ESTUDIANTE]: Profe, ¿Cuál es la fórmula de recurrencia del método de Bisección?
[TUTOR IA]:   </think>Esystem
Eres un chatbot del curso de Métodos Numéricos en Ingenieria. Respondes preguntas de estudiantes sobre la guia de estudio.<|box_end|>J
</think>Euser
Profe, ¿Cuál es la fórmula de recurrencia del método de Bisección?<|box_end|>J
</think>Eassistant
Se define la función f(x), se toman dos valores, x0 y x1, tales que f(x0) y f(x1) se cambian de signo y se define una variable, x2 = (x0 + x1)/2. La función f(x2) se evalúa para verificar si el cambio de signo ocurre en x2. Si ocurre, se define x0 = x1 y x1 = x2, o viceversa, según sea el signo de f(x2).<|box_end|>J
--------------------------------------------------------------------------------

[ESTUDIANTE]: ¿Para qué sirven los Subprogramas?
[TUTOR IA]:   </think>Esystem
Eres un chatbot del curso de Métod